In [13]:
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from pathlib import Path

In [14]:
# Define o diretório base
outdir = Path("/home/cavalheiromf/doutorado/data/sequences")

In [15]:
def translate_cds_to_protein(input_fasta, output_fasta, table=1):
    protein_records = []
    
    print(f"Lendo CDS de {input_fasta}...")
    
    for record in SeqIO.parse(input_fasta, "fasta"):
        # Traduz a sequência de DNA para Proteína
        # to_stop=True remove o asterisco (*) do stop codon final
        protein_seq = record.seq.translate(table=table, to_stop=True)
        
        # Cria um novo registro mantendo o ID original
        prot_record = SeqRecord(
            protein_seq, 
            id=record.id, 
            description=record.description
        )
        protein_records.append(prot_record)
    
    # Salva o resultado
    SeqIO.write(protein_records, output_fasta, "fasta")
    print(f"Sucesso! {len(protein_records)} proteínas salvas em {output_fasta}")

In [16]:
# Define os arquivos usando o operador / para unir os caminhos
input_path = outdir / "Vper.hapA.v1.softmasked.cds.fasta"
output_path = outdir / "Vper.hapA.v1.softmasked.proteins.fasta"

# table=1 é o padrão (Eucariotos), use table=11 para Bactérias/Archaea
translate_cds_to_protein(str(input_path), str(output_path), table=1)

Lendo CDS de /home/cavalheiromf/doutorado/data/sequences/Vper.hapA.v1.softmasked.cds.fasta...
Sucesso! 26466 proteínas salvas em /home/cavalheiromf/doutorado/data/sequences/Vper.hapA.v1.softmasked.proteins.fasta


/home/cavalheiromf/miniforge3/envs/jupyterEnv/lib/python3.14/site-packages/Bio/Seq.py:2877: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


In [17]:
# Define os arquivos usando o operador / para unir os caminhos
input_path = outdir / "Vtub.hapA.v1.softmasked.cds.fasta"
output_path = outdir / "Vtub.hapA.v1.softmasked.proteins.fasta"

# Execução
# table=1 é o padrão (Eucariotos), use table=11 para Bactérias/Archaea
translate_cds_to_protein(str(input_path), str(output_path), table=1)

Lendo CDS de /home/cavalheiromf/doutorado/data/sequences/Vtub.hapA.v1.softmasked.cds.fasta...
Sucesso! 25551 proteínas salvas em /home/cavalheiromf/doutorado/data/sequences/Vtub.hapA.v1.softmasked.proteins.fasta


## Encontrando possíveis OSCAs

In [18]:
import subprocess
from pathlib import Path
import os

## Hmmer

In [19]:
def run_hmmsearch(proteins_fasta, pfam_hmm, outdir):
    # 1. Converter caminhos para objetos Path
    proteins_fasta = Path(proteins_fasta)
    pfam_hmm = Path(pfam_hmm)
    outdir = Path(outdir)
    
    # 2. Verificar se o arquivo HMM está compactado (.gz)
    # O hmmsearch NÃO lê .gz. Vamos descompactar se necessário.
    if pfam_hmm.suffix == ".gz":
        print(f"Descompactando {pfam_hmm}...")
        subprocess.run(["gunzip", "-f", str(pfam_hmm)], check=True)
        # Atualiza o caminho para o arquivo sem o .gz
        pfam_hmm = pfam_hmm.with_suffix("") 

    # 3. Definir o arquivo de saída da tabela
    output_table = outdir / f"{proteins_fasta.stem}.hmmout.txt"

    # 4. Montar o comando exatamente como o HMMER exige
    # [comando, opções, arquivos_posicionais]
    cmd = [
        "/home/cavalheiromf/miniforge3/envs/jupyterEnv/bin/hmmsearch",
        "--tblout", str(output_table),  # Opção de saída em tabela
        "--cpu", "4",                   # Uso de CPUs
        str(pfam_hmm),                  # 1º argumento posicional
        str(proteins_fasta)             # 2º argumento posicional
    ]

    print(f"Executando comando:\n{' '.join(cmd)}\n")

    try:
        # Executa o comando
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        print(f"Sucesso! Resultados salvos em: {output_table}")
        
    except subprocess.CalledProcessError as e:
        print("--- ERRO NO HMMSEARCH ---")
        print(f"Comando: {e.cmd}")
        print(f"Erro de saída (stderr):\n{e.stderr}")
        print(f"Saída padrão (stdout):\n{e.stdout}")

In [20]:
# --- Parâmetros ---
base_dir = "/home/cavalheiromf/doutorado/data/sequences"
out_dir = "/home/cavalheiromf/doutorado/analysis/hmmer"
input_prot = os.path.join(base_dir, "Vper.hapA.v1.softmasked.proteins.fasta")
input_hmm = os.path.join(base_dir, "PF02714.hmm") # Verifique se o nome está correto agora

run_hmmsearch(input_prot, input_hmm, out_dir)

Executando comando:
/home/cavalheiromf/miniforge3/envs/jupyterEnv/bin/hmmsearch --tblout /home/cavalheiromf/doutorado/analysis/hmmer/Vper.hapA.v1.softmasked.proteins.hmmout.txt --cpu 4 /home/cavalheiromf/doutorado/data/sequences/PF02714.hmm /home/cavalheiromf/doutorado/data/sequences/Vper.hapA.v1.softmasked.proteins.fasta

Sucesso! Resultados salvos em: /home/cavalheiromf/doutorado/analysis/hmmer/Vper.hapA.v1.softmasked.proteins.hmmout.txt


In [22]:
# --- Parâmetros ---
base_dir = "/home/cavalheiromf/doutorado/data/sequences"
out_dir = "/home/cavalheiromf/doutorado/analysis/hmmer"
input_prot = os.path.join(base_dir, "Vtub.hapA.v1.softmasked.proteins.fasta")
input_hmm = os.path.join(base_dir, "PF02714.hmm") # Verifique se o nome está correto agora

run_hmmsearch(input_prot, input_hmm, base_dir)

Executando comando:
/home/cavalheiromf/miniforge3/envs/jupyterEnv/bin/hmmsearch --tblout /home/cavalheiromf/doutorado/data/sequences/Vtub.hapA.v1.softmasked.proteins.hmmout.txt --cpu 4 /home/cavalheiromf/doutorado/data/sequences/PF02714.hmm /home/cavalheiromf/doutorado/data/sequences/Vtub.hapA.v1.softmasked.proteins.fasta

Sucesso! Resultados salvos em: /home/cavalheiromf/doutorado/data/sequences/Vtub.hapA.v1.softmasked.proteins.hmmout.txt


## Extraindo sequência proteica

In [25]:
import pandas as pd

In [69]:
vper = pd.read_table('/home/cavalheiromf/doutorado/analysis/hmmer/Vper.hapA.v1.softmasked.proteins.hmmout.txt', 
comment='#', header = None, sep = '-')
oscaVper = vper[0].to_list()
oscaVper = [x.strip() for x in oscaVper]

vtub = pd.read_table('/home/cavalheiromf/doutorado/analysis/hmmer/Vtub.hapA.v1.softmasked.proteins.hmmout.txt', 
comment='#', header = None, sep = '-')
oscaVtub = vtub[0].to_list()
oscaVtub = [x.strip() for x in oscaVtub]

In [70]:
# Exemplo: wanted como lista/set de IDs
wanted = oscaVper  # vtub deve ser iterável (lista, série, etc.)

in_fa  = "/home/cavalheiromf/doutorado/data/sequences/Vper.hapA.v1.softmasked.proteins.fasta"
out_fa = "/home/cavalheiromf/doutorado/data/sequences/Vper.osca.fasta"

n = 0
with open(out_fa, "w") as out:
    for rec in SeqIO.parse(in_fa, "fasta"):
        if rec.id in wanted:
            SeqIO.write(rec, out, "fasta")
            n += 1

print("Salvas:", n)


Salvas: 14


In [ ]:
# Exemplo: wanted como lista/set de IDs
wanted = oscaVtub  # vtub deve ser iterável (lista, série, etc.)

in_fa  = "/home/cavalheiromf/doutorado/data/sequences/Vtub.hapA.v1.softmasked.proteins.fasta"
out_fa = "/home/cavalheiromf/doutorado/data/sequences/Vtub.osca.fasta"

n = 0
with open(out_fa, "w") as out:
    for rec in SeqIO.parse(in_fa, "fasta"):
        if rec.id in wanted:
            SeqIO.write(rec, out, "fasta")
            n += 1

print("Salvas:", n)


Salvas: 13


## Juntando arquivos fasta

In [73]:
!cd /home/cavalheiromf/doutorado/data/sequences && cat otherOSCAs.fasta Vtub.osca.fasta Vper.osca.fasta > allOSCAs.fasta

## Rodando alinhamento

In [79]:
mafft = Path("/home/cavalheiromf/miniforge3/envs/jupyterEnv/bin/mafft")  # ajuste se necessário
inp   = Path("/home/cavalheiromf/doutorado/data/sequences/allOSCAs.fasta")
out   = Path("/home/cavalheiromf/doutorado/analysis/mafft/allOSCAs.aligned.fasta")

out.parent.mkdir(parents=True, exist_ok=True)

cmd = [
    str(mafft),
    "--auto",
    "--thread", "32", 
    str(inp)
]

with open(out, "w") as fout:
    proc = subprocess.run(cmd, stdout=fout, stderr=subprocess.PIPE, text=True)

print(proc.stderr)      # log do MAFFT
proc.check_returncode() # levanta erro se falhar
print("Alinhamento salvo em:", out)


outputhat23=16
treein = 0
compacttree = 0
stacksize: 8192 kb
rescale = 1
All-to-all alignment.
    0 / 76 (by thread   2) 
   10 / 76 (by thread  31) 
   20 / 76 (by thread   4) 
   30 / 76 (by thread   6) 
   40 / 76 (by thread  24) 
   50 / 76 (by thread   6) 
   60 / 76 (by thread  13) 
   70 / 76 (by thread  25) 
tbfast-pair (aa) Version 7.526
alg=L, model=BLOSUM62, 2.00, -0.10, +0.10, noshift, amax=0.0
32 thread(s)

outputhat23=16
Loading 'hat3.seed' ... 
done.
Writing hat3 for iterative refinement
rescale = 1
Gap Penalty = -1.53, +0.00, +0.00
tbutree = 1, compacttree = 0
Constructing a UPGMA tree ... 

    0 / 76
   10 / 76
   20 / 76
   30 / 76
   40 / 76
   50 / 76
   60 / 76
   70 / 76
done.

Progressive alignment ... 

STEP     1 /75 (thread    0) 
STEP     2 /75 (thread    1) 
STEP     3 /75 (thread    2) 
STEP     4 /75 (thread    4) 
STEP     6 /75 (thread    6) 
STEP     7 /75 (thread    5) 
STEP     8 /75 (thread    7) 
STEP     9 /75 (thread    8) 
STEP    10 /75 (threa

## Rodando árvore filogenética

In [82]:
iqtree = Path("/home/cavalheiromf/miniforge3/envs/jupyterEnv/bin/iqtree3")
aln    = Path("/home/cavalheiromf/doutorado/analysis/mafft/allOSCAs.aligned.fasta")
outpre = Path("/home/cavalheiromf/analysis/iqtree")


outpre.parent.mkdir(parents=True, exist_ok=True)

cmd = [
    str(iqtree),
    "-s", str(aln),
    "-st", "AA",      # garante que é aminoácido
    "-m", "MFP",      # ModelFinder Plus (seleciona modelo automaticamente)
    "-nt", "4",       # threads (ou use "AUTO")
    "-pre", str(outpre),
]

proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
print(proc.stderr)
proc.check_returncode()


IQ-TREE version 3.0.1 for Linux x86 64-bit built Jul  9 2025
Developed by Bui Quang Minh, Thomas Wong, Nhan Ly-Trong, Huaiyan Ren
Contributed by Lam-Tung Nguyen, Dominik Schrempf, Chris Bielow,
Olga Chernomor, Michael Woodhams, Diep Thi Hoang, Heiko Schmidt

Host:    dayhoff-omics (AVX2, FMA3, 15 GB RAM)
Command: /home/cavalheiromf/miniforge3/envs/jupyterEnv/bin/iqtree3 -s /home/cavalheiromf/doutorado/analysis/mafft/allOSCAs.aligned.fasta -st AA -m MFP -nt 4 -pre /home/cavalheiromf/analysis/iqtree
Seed:    385397 (Using SPRNG - Scalable Parallel Random Number Generator)
Time:    Wed Jan 21 13:49:44 2026
Kernel:  AVX+FMA - 4 threads (32 CPU cores detected)

Reading alignment file /home/cavalheiromf/doutorado/analysis/mafft/allOSCAs.aligned.fasta ... Fasta format detected
Reading fasta file: done in 0.00167184 secs
Alignment most likely contains protein sequences
Constructing alignment: done in 0.00120399 secs using 99.92% CPU
Alignment has 76 sequences with 1697 columns, 1190 distinct p

## Renomeando árvore filogenética

In [90]:
awk_prog = r"""
NR==FNR { map[$1]=$2; next }
{
  for (k in map) {
    gsub(k ":", map[k] "§:")
  }
  gsub("§", "")
  print
}
"""

map_tsv = "/home/cavalheiromf/doutorado/data/annotation/oscasIDs.txt"
in_nwk  = "/home/cavalheiromf/doutorado/analysis/iqtree/iqtree.treefile"
out_nwk = "/home/cavalheiromf/doutorado/analysis/iqtree/iqtree.renamed.treefile"

with open(out_nwk, "w", encoding="utf-8") as fout:
    subprocess.run(
        ["awk", "-F\t", awk_prog, map_tsv, in_nwk],
        check=True,
        stdout=fout,
        text=True
    )


## Identificando domínios nos OSCAs

In [137]:
mapID = pd.read_table(
    "/home/cavalheiromf/doutorado/data/annotation/oscasIDs.txt",
    header=None,
    names=["id", "gene"]  # já nomeia as 2 colunas
)

mapping = dict(zip(mapID["id"], mapID["gene"]))

In [154]:
df = pd.read_table('/home/cavalheiromf/doutorado/analysis/interproscan/iprscan5-R20260121-182814-0314-59205818-p1m.tsv', header=None)
df_filtered = df[df[3] == 'Pfam']
dom = df_filtered[[0,6,7,4]].copy()
dom.columns = ['id', 'start', 'end', 'domain']
# dom['id'] = dom['id'].map(mapping)
dom.to_csv('/home/cavalheiromf/doutorado/analysis/interproscan/Vtub.osca.txt', sep = '\t', index=False)

In [155]:
df = pd.read_table('/home/cavalheiromf/doutorado/analysis/interproscan/iprscan5-R20260121-182837-0471-82132136-p1m.tsv', header=None)
df_filtered = df[df[3] == 'Pfam']
dom = df_filtered[[0,6,7,4]].copy()
dom.columns = ['id', 'start', 'end', 'domain']
# dom['id'] = dom['id'].map(mapping)
dom.to_csv('/home/cavalheiromf/doutorado/analysis/interproscan/Vper.osca.txt', sep = '\t', index=False)